# Aspire : l'asynchrone aux frontieres natives - P/Invoke, UnmanagedCallersOnly, et la famine observee

## Pourquoi ce notebook existe

Le curriculum Aspire enseigne deja six analyseurs Roslyn (`AgentGuard.Analyzers`) qui detectent les blessures classiques du code d'agent manage : `TaskResultBlock` (`.Result` qui bloque), `AsyncVoid` (exception non observee), `SyncOverAsync` (`.GetAwaiter().GetResult()` masque en sync), `TaskRunFire` (`Task.Run` qui fire-and-forget), `CancellationTokenPropagation` (annulation perdue), et `SyncOverAsyncConfigureAwait` (`ConfigureAwait(false)` mal place). Ces regles vivent **dans la compilation**, et c'est precisement leur force.

Mais toute la track .NET consomme des **bibliotheques natives** - Z3, ONNX Runtime, ML.NET, Infer.NET, IKVM/Tweety - et aucune de ces frontieres n'est enseignee. Un driver C# sur un moteur Rust (Tokio) traverse une frontiere d'executeur : la continuation .NET s'execute *sur un thread qui n'appartient pas au thread-pool manage*, et la meme physique async qui produisait un deadlock en manage produit ici une **famine du pool etranger** - detectable, mesurable, reparable par les memes leviers (`TaskCompletionSource` + `RunContinuationsAsynchronously`).

L'article ScyllaDB du 2026-08-31 (*Self-Baked Async FFI Framework for Rust <-> C# Interop*) est la demonstration *production* de cette physique. Ce notebook en distille le cote .NET - **sans installer Rust**, conformement a la decision user du 2026-09-01 : la pile Rust n'est pas dans la stack, et le cote Tokio est simule par un pool de threads dedies a taille fixe (simulation disclose honnetement).

## Table de correspondance - le fil narratif

Les garde-fous appris en manage sont les **memes lois**, a la frontiere. Chaque ligne de la table est reprise par une section plus bas.

| AgentGuard (06-Aspire-GardeFous) | Contrepartie frontiere native (article ScyllaDB) | Meme physique |
|---|---|---|
| `TaskResultBlockAnalyzer` - `.Result` bloque un thread du pool -> deadlock | Callback de completion Tokio qui execute la continuation .NET **sur le thread Tokio** -> famine du pool etranger a taille fixe | Continuation synchrone sur un executeur qu'on ne possede pas |
| Le fix enseigne : `await` | Le fix de l'article : `TaskCompletionSource` + `RunContinuationsAsynchronously` | Forcer la continuation a sauter sur le bon pool |
| `AsyncVoidAnalyzer` - exception non observee = crash | Une exception qui traverse l'FFI = UB - attrapee et marshalee en callback d'echec | L'erreur s'achemine par le canal prevu, jamais laissee fuir |
| `CancellationTokenPropagationAnalyzer` | Aucune annulation ne traverse l'FFI sans plomberie explicite (le TCB n'a que success/fail) | L'annulation est une responsabilite du pont |
| `using`/`IDisposable` (duree de vie, theme EF Core) | `SafeHandle` + `GCHandle` + comptage de refs atomique | La duree de vie traverse une frontiere d'ownership |

Le reste du notebook reprend cette table dans le code, section par section.

**Note d'environnement** : les demonstrations de P/Invoke et reverse P/Invoke utilisent `[DllImport]` (reflexion runtime, ~30 ns/overhead) pour rester compatibles avec le contexte .NET Interactive. En production, `[LibraryImport]` (.NET 7+, source-genere, ~3 ns/overhead) est preferable - le source-generation reclame un contexte `partial class` que le kernel Jupyter ne fournit pas par cellule.

## Section 1 - P/Invoke direct : types blittables et conventions d'appel

Le runtime .NET expose deux API pour invoquer du code natif : `[DllImport]` (reflexion a l'execution) et `[LibraryImport]` (.NET 7+, source-genere a la compilation, plus rapide). La cible pedagogique ici : `EnumWindows` de `user32.dll`, quiitere toutes les fenetres de premier niveau et appelle un callback pour chacune.

**Types blittables** : un type est *blittable* si sa representation memoire est identique en C# et en C. Les primitifs (`int`, `long`, `double`, `byte`), les `struct` sequentiels de primitifs, et les pointeurs sont blittables. Les `string`, les `bool` (sujet piege §6), et les `class` ne le sont **pas** : ils necessitent un marshaling explicite.

In [1]:
using System.Runtime.InteropServices;

// Delegue manage stable pour le callback (pre-.NET 5 style).
// En .NET 5+, [UnmanagedCallersOnly] sur une methode static est preferable,
// mais demande un contexte partial class hors Jupyter.
delegate bool EnumWindowsProc(IntPtr hWnd, IntPtr lParam);

// P/Invoke via reflexion runtime - fonctionne dans tous les contextes.
[DllImport("user32.dll")]
[return: MarshalAs(UnmanagedType.Bool)]
static extern bool EnumWindows(EnumWindowsProc enumProc, IntPtr lParam);

Console.WriteLine("P/Invoke defini : user32.dll!EnumWindows.");
Console.WriteLine("Le retour est annote [MarshalAs(UnmanagedType.Bool)] (4 octets Win32).");
Console.WriteLine("La cible user32.dll est dans System32 sur Windows.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

P/Invoke defini : user32.dll!EnumWindows.


Le retour est annote [MarshalAs(UnmanagedType.Bool)] (4 octets Win32).


La cible user32.dll est dans System32 sur Windows.


## Section 2 - Reverse P/Invoke : le callback manage appele depuis le natif

Le callback que `EnumWindows` attend est un **pointeur de fonction C**, qu'on materialise via un `delegate` manage. En .NET 5+, on utiliserait `[UnmanagedCallersOnly]` pour avoir un `delegate* unmanaged` direct (plus rapide, duree de vie geree par le caller), mais le kernel Jupyter ne fournit pas un contexte `partial class` propre - d'ou l'usage du delegue manage ici, qui est la version pre-.NET 5 (toujours supportee).

**Le piege** : si le GC collecte le delegue pendant que le natif detient son pointeur, le callback appelle dans le vide. Le `GC.KeepAlive` a la fin est ce qui empeche ca.

In [2]:
// Le callback : recoit (hwnd, lParam).
// lParam transporte un pointeur vers notre compteur (pattern Win32 standard).
// Retour : true = continuer l'iteration, false = stopper.
static bool EnumWindowsCallbackImpl(IntPtr hWnd, IntPtr lParam)
{
    unsafe { (*(int*)lParam)++; }
    return true;
}

// Invocation reelle via P/Invoke.
// `enumProc` est un delegate manage : .NET le marshale en pointeur C au passage de la frontiere.
int counter = 0;
EnumWindowsProc enumProc = EnumWindowsCallbackImpl;
unsafe
{
    fixed (int* pCounter = &counter)
    {
        bool ok = EnumWindows(enumProc, (IntPtr)pCounter);
        Console.WriteLine($"EnumWindows ok={ok}");
        Console.WriteLine($"Fenetres de premier niveau enumerees = {counter}");
    }
}

// Barriere GC : tant qu'on n'a pas atteint ce point, le delegate reste vivant.
GC.KeepAlive(enumProc);

EnumWindows ok=True


Fenetres de premier niveau enumerees = 376


## Section 3 - Le pont async : `TaskCompletionSource` + `GCHandle`

Le natif ne sait pas ce qu'est une `Task<T>`. Pour exposer une operation **async** au code manage, on construit un `TaskCompletionSource<T>` (TCS) en C#, on passe son **handle** au natif via un struct C-compatible (le *TCB*, l'equivalent du callback-handle de ScyllaDB), et le natif appelle nos deux pointeurs de fonction (`success` et `fail`) quand l'operation termine.

**Le piege GC** : `GCHandle.Alloc(tcs, GCHandleType.Pinned)` epingle le TCS en memoire (le natif detient son adresse). Sans ca, le GC pourrait deplacer l'objet et le pointeur qu'on a passe deviendrait invalide.

**Le second piege** : si on `SetResult` *sur le thread natif*, la continuation s'execute **sur le thread natif**, pas sur le thread-pool manage. C'est precisement la famine ScyllaDB - le fix est `RunContinuationsAsynchronously`, detaille en §4.

In [3]:
using System.Runtime.CompilerServices;
using System.Threading.Tasks;

// TCB = la struct C-compatible qu'on passerait au natif.
// Deux pointeurs de fonction : success et fail. Le runtime les appelle.
[StructLayout(LayoutKind.Sequential)]
struct AsyncCallbackHandle
{
    public IntPtr SuccessCallback;
    public IntPtr FailCallback;
    public IntPtr UserData;
}

// Constructeur de pont : cree le TCS, l'attache en GCHandle Normal, et retourne le handle.
// Note : on utilise GCHandleType.Normal (pas Pinned) car TaskCompletionSource<T> n'est pas
// blittable (il contient des references internes). En production Rust/C, le code natif
// epinglerait le bloc memoire ou recevrait un pointeur opaque (cookie) qu'il rappellerait
// pour resoudre vers le TCS. Ici, on montre le cas pedagogique : le handle garde l'objet
// vivant tant qu'on n'a pas appele Free().
// Note 2 : en code de production, on utiliserait un wrapper IDisposable pour garantir
// le Free() meme en cas d'exception. Ici, demo courte.
TaskCompletionSource<int> tcs = new(
    TaskCreationOptions.RunContinuationsAsynchronously);
GCHandle pin = GCHandle.Alloc(tcs); // Normal handle (pas Pinned : TCS non-blittable)
IntPtr rawHandle = (IntPtr)pin; // l'opaque handle lui-meme

Console.WriteLine($"TCS attache (GCHandle opaque) = 0x{rawHandle.ToInt64():X}");
Console.WriteLine("RunContinuationsAsynchronously = active (anti-famine §4).");
Console.WriteLine("Le TCS resterait vivant tant que `pin` n'est pas Free().");

// Demonstration : on peut recuperer le TCS depuis le handle opaque.
TaskCompletionSource<int> tcsBack = (TaskCompletionSource<int>)pin.Target!;
Console.WriteLine($"Round-trip TCS recuperable : {object.ReferenceEquals(tcs, tcsBack)}");

// Cleanup obligatoire.
pin.Free();

TCS attache (GCHandle opaque) = 0x27804CD15E8


RunContinuationsAsynchronously = active (anti-famine §4).


Le TCS resterait vivant tant que `pin` n'est pas Free().


Round-trip TCS recuperable : True


## Section 4 - La famine, **mesurée**

La version précédente de cette section observait l'identité du thread qui exécute la continuation, et constatait que les deux cas se ressemblaient : `ContinueWith` **sans** `TaskContinuationOptions.ExecuteSynchronously` planifie toujours la continuation sur le pool managé. La discrimination était donc impossible par construction, et la cellule ne pouvait que constater son propre échec.

Cette section la mesure sur la métrique qui compte en production : **le temps que le thread natif passe bloqué dans `SetResult`**. Pendant ce temps, ce thread ne dispatche pas ses propres entrées-sorties -> c'est la famine du pool étranger, chiffrée.

Deux leviers sont exposés, et ils ne se ressemblent pas :

- **`TaskContinuationOptions.ExecuteSynchronously`** : autorise l'exécution **en ligne** de la continuation sur le thread qui complète la tâche. C'est le levier qui rend la famine possible.
- **`TaskCreationOptions.RunContinuationsAsynchronously`** : force la continuation à partir sur le pool managé, même quand l'exécution en ligne est demandée. C'est le levier qui protège - le thread natif est libéré immédiatement.

Le travail .NET est fait par la **continuation** (c'est le rôle du handler côté .NET) : c'est elle qui, exécutée en ligne, occupe le thread natif. Le thread natif, lui, ne fait qu'appeler `SetResult` -> c'est le rôle du thread Tokio.

In [4]:
using System.Diagnostics;
using System.Threading;
using System.Threading.Tasks;

// La famine, MESUREE.
//
// La metrique n'est pas l'identite du thread de la continuation : c'est le
// temps que le thread NATIF passe BLOQUE dans `SetResult`. Pendant ce temps,
// ce thread ne dispatche pas ses propres entrees-sorties -> c'est la famine
// du pool etranger, chiffree.
//
// Deux leviers, absents de la version precedente de cette cellule :
//
// 1. `TaskContinuationOptions.ExecuteSynchronously` autorise l'execution EN
//    LIGNE de la continuation sur le thread qui complete la tache. Sans
//    cette option, `ContinueWith` planifie TOUJOURS la continuation sur le
//    pool manage : la discrimination etait impossible par construction, et
//    la cellule ne pouvait que constater son propre echec.
// 2. `TaskCreationOptions.RunContinuationsAsynchronously` force la
//    continuation a partir sur le pool manage, meme quand l'execution en
//    ligne est demandee : c'est LE levier qui protege - le thread natif est
//    libere immediatement.
//
// C'est la CONTINUATION qui fait le travail .NET (le role du handler cote
// .NET) : c'est elle qui, executee en ligne, occupe le thread natif.

double Ms(long ticks) => ticks * 1000.0 / Stopwatch.Frequency;

// Le travail .NET de la continuation : un calcul long, stand-in pour le
// handler cote .NET qui fait ses traitements sur le thread natif.
double TravailNet(int tours)
{
    double acc = 0;
    for (int i = 0; i < tours; i++) acc += Math.Sqrt(i);
    GC.KeepAlive(acc);
    return acc;
}

// Mesure repetee : 5 runs par configuration, mediane + min/max (#21523).
// La mesure unique est une illustration ; la repetition la rend citable.
// Chaque run rend aussi les identifiants de thread (natif / reprise) : la
// « Lecture du resultat » les cite, et ils ancrent le mecanisme.

(double ms, int idNatif, int idReprise) MesurerTempsBlocage(bool runAsync, int tours)
{
    var tcs = runAsync
        ? new TaskCompletionSource<int>(TaskCreationOptions.RunContinuationsAsynchronously)
        : new TaskCompletionSource<int>();

    var sw = Stopwatch.StartNew();
    long tEntree = 0, tSortie = 0;
    int idNatif = -1, idReprise = -1;
    var finReprise = new ManualResetEventSlim(false);

    tcs.Task.ContinueWith(
        _ =>
        {
            idReprise = Thread.CurrentThread.ManagedThreadId;
            TravailNet(tours);
            finReprise.Set();
        },
        CancellationToken.None,
        TaskContinuationOptions.ExecuteSynchronously,
        TaskScheduler.Default);

    var natif = new Thread(() =>
    {
        idNatif = Thread.CurrentThread.ManagedThreadId;
        tEntree = sw.ElapsedTicks;
        tcs.TrySetResult(1);
        tSortie = sw.ElapsedTicks;
    }) { IsBackground = true, Priority = ThreadPriority.BelowNormal };

    natif.Start();
    natif.Join();
    finReprise.Wait(TimeSpan.FromSeconds(10));
    return (Ms(tSortie - tEntree), idNatif, idReprise);
}

const int N_RUNS = 5;
const int TOURS = 20_000_000;

var tempsA = new List<double>();
var tempsB = new List<double>();
var idsA = new List<(int natif, int reprise)>();
var idsB = new List<(int natif, int reprise)>();

Console.WriteLine("=== La famine, mesuree (" + N_RUNS + " runs par configuration) ===");
Console.WriteLine();

for (int run = 0; run < N_RUNS; run++)
{
    var (msA, natifA, repriseA) = MesurerTempsBlocage(false, TOURS);
    var (msB, natifB, repriseB) = MesurerTempsBlocage(true, TOURS);
    tempsA.Add(msA);
    tempsB.Add(msB);
    idsA.Add((natifA, repriseA));
    idsB.Add((natifB, repriseB));
}

void Rapporter(string libelle, List<double> temps)
{
    var tries = new List<double>(temps);
    tries.Sort();
    var runs = string.Join(", ", temps.ConvertAll(t => t.ToString("F2")));
    Console.WriteLine(libelle);
    Console.WriteLine($"    runs     = [{runs}] ms");
    Console.WriteLine($"    mediane  = {tries[tries.Count / 2]:F2} ms");
    Console.WriteLine($"    min/max  = {tries[0]:F2} / {tries[tries.Count - 1]:F2} ms");
    Console.WriteLine();
}

// Les identifiants de thread des 5 runs : sous [A], la reprise s'execute EN
// LIGNE sur le thread natif (identifiants egaux) ; sous [B], elle part sur
// le pool manage (identifiants differents) - le symptome lisible du temps.
void RapporterIdentites(string libelle, List<(int natif, int reprise)> ids)
{
    var paires = new List<string>();
    bool tousIdentiques = true;
    foreach (var (natif, reprise) in ids)
    {
        paires.Add($"{natif}/{reprise}");
        if (natif != reprise) tousIdentiques = false;
    }
    Console.WriteLine($"{libelle} -- threads natif/reprise");
    Console.WriteLine($"    [{string.Join(", ", paires)}]");
    Console.WriteLine(tousIdentiques
        ? "    -> identiques : la reprise s'execute en ligne sur le thread natif."
        : "    -> differents : la reprise part sur le pool manage.");
    Console.WriteLine();
}

Rapporter("[A] ExecuteSynchronously, SANS RunContinuationsAsynchronously", tempsA);
RapporterIdentites("[A] ExecuteSynchronously, SANS RunContinuationsAsynchronously", idsA);
Rapporter("[B] ExecuteSynchronously, AVEC RunContinuationsAsynchronously", tempsB);
RapporterIdentites("[B] ExecuteSynchronously, AVEC RunContinuationsAsynchronously", idsB);

Console.WriteLine("Les deux configurations different de plusieurs ordres de grandeur : c'est la famine.");
Console.WriteLine("A : le thread natif fait le travail .NET a la place de dispatcher ses I/O.");
Console.WriteLine("B : RunContinuationsAsynchronously renvoie la continuation au pool manage,");
Console.WriteLine("    et le thread natif est libere immediatement.");

=== La famine, mesuree (5 runs par configuration) ===


[A] ExecuteSynchronously, SANS RunContinuationsAsynchronously


    runs     = [171,09, 206,23, 153,49, 181,81, 137,07] ms


    mediane  = 171,09 ms


    min/max  = 137,07 / 206,23 ms


...

    -> differents : la reprise part sur le pool manage.


Les deux configurations different de plusieurs ordres de grandeur : c'est la famine.


A : le thread natif fait le travail .NET a la place de dispatcher ses I/O.


B : RunContinuationsAsynchronously renvoie la continuation au pool manage,


    et le thread natif est libere immediatement.


**Lecture du résultat** :

- **Sans `RunContinuationsAsynchronously`** : la reprise s'exécute **en ligne sur le thread natif** (les deux identifiants de thread sont identiques) et le thread natif reste **bloqué pendant tout le travail .NET** de la continuation. C'est la famine : en production, ce thread Tokio dispatche des requêtes d'entrée-sortie à grande cadence, et le temps qu'il passe ici lui est soustrait.
- **Avec `RunContinuationsAsynchronously`** : la reprise part sur le **pool managé** (identifiants de thread différents) et le thread natif est **libéré immédiatement** -> le temps de blocage mesuré retombe à celui d'une simple notification.

La différence d'identité de thread **est** la différence, mais elle n'en est qu'un **symptôme** : ce que la mesure quantifie, c'est le **temps** que le thread natif perd. C'est ce temps, et non une identité, qui se soustrait au pool étranger.

Ce que l'ancienne version de cette section ne pouvait pas faire : `ContinueWith` **sans** `ExecuteSynchronously` planifie toujours sur le pool managé, donc les deux identifiants de thread étaient identiques et la discrimination était impossible. C'est le même mécanisme que celui que l'analyseur `TaskResultBlock` détecte côté managé : un `.Result` qui bloque fait attendre le pool au lieu de lui rendre la main.

## Section 5 - Durees de vie : `SafeHandle`, `GCHandle`, `GC.KeepAlive`

Quand un objet manage traverse une frontiere d'ownership, le ramasse-miettes ne peut pas deviner qui le possede de l'autre cote. Trois mecanismes pour ca :

- **`SafeHandle`** : enveloppe un pointeur natif (HANDLE, FILE*, etc.) avec un `ReleaseHandle()` finalizer-safe. C'est l'equivalent d'un `unique_ptr<T, Deleter>` C++. Le handle est recyclable : `Close()` ne libere pas immediatement, il retourne au pool.
- **`GCHandle`** : on l'a vu en §3. `Alloc(obj)` empeche le GC de collecter l'objet, `Pin()` empeche le GC de le deplacer en memoire. **Attention** : un `GCHandle` qui fuit epingle l'objet pour la vie du processus.
- **`GC.KeepAlive(obj)`** : une *barriere* pour le JIT. Le JIT peut decider que `obj` est mort apres la derniere utilisation syntaxique - si natif detient encore un pointeur, c'est un bug. `KeepAlive` force le GC a considerer `obj` vivant jusqu'a ce point.

**Piege classique** : passer une chaine managee a `Marshal.StringToHGlobalUni`, oublier `FreeHGlobal`, et la memoire fuit. Le pattern correct : `try/finally`.

In [5]:
using System.Runtime.InteropServices;

// Demonstration du pattern try/finally sur AllocHGlobal/FreeHGlobal.
// C'est l'anti-pattern n°1 des appels FFI .NET - toujours mettre dans un try/finally.
IntPtr buffer = Marshal.AllocHGlobal(64);
try
{
    byte[] data = new byte[64];
    for (int i = 0; i < 64; i++) data[i] = (byte)i;
    Marshal.Copy(data, 0, buffer, 64);
    Console.WriteLine($"Buffer alloue a 0x{buffer.ToInt64():X}, 64 octets remplis.");
}
finally
{
    Marshal.FreeHGlobal(buffer);
    Console.WriteLine("Buffer libere proprement dans le finally.");
}

// GC.KeepAlive est implicite ici (les variables locales restent vivantes jusqu'a la fin de la cellule),
// mais en cas de callback asynchrone, il faudrait un KeepAlive explicite apres l'appel natif.

Buffer alloue a 0x2B8A6652740, 64 octets remplis.


Buffer libere proprement dans le finally.


## Section 6 - Marshaling blittable, et le **piege du `bool`**

Un `bool` C# fait **1 octet** en memoire. Un `bool` C / `BOOL` Win32 fait **4 octets** (`int`). Si on marshale un `bool` C# vers du natif sans precaution, la convention d'appel recoit 1 octet la ou le natif en lit 4 - **donc le natif lit 3 octets qui ne lui appartiennent pas** (registre ou pile). Sur x86_64 Windows c'est rarement fatal, mais c'est de l'UB certifiee.

**Le fix** : `[MarshalAs(UnmanagedType.Bool)]` pour la convention Win32 (4 octets), `[MarshalAs(UnmanagedType.U1)]` pour un struct partage C qui veut explicitement 1 octet.

**Le piege des structs non sequentiels** : `[StructLayout(LayoutKind.Sequential)]` est la valeur par defaut, mais le JIT peut *reordonner* les champs pour minimiser le padding. Si le code C attend un ordre strict, il faut `[StructLayout(LayoutKind.Sequential, Pack = 1)]` pour desactiver le padding.

**`ReadOnlySpan<byte>` zero-copie** : depuis .NET 6, on peut marshaller un `(byte* ptr, nuint len)` C en `ReadOnlySpan<byte>` manage sans copie - c'est ce que ScyllaDB appelle `FFISlice`.

In [6]:
using System.Runtime.InteropServices;

// Struct C-compatible : ordre strict, pack = 4 (Win32 default).
[StructLayout(LayoutKind.Sequential, Pack = 4)]
struct FFISlice
{
    public IntPtr Ptr;
    public UIntPtr Len;
}

// Le piege du bool : sans annotation explicite, le marshaling par defaut
// de C# est 1 octet. Win32 BOOL est 4 octets. -> UB garantie.
[DllImport("user32.dll")]
[return: MarshalAs(UnmanagedType.Bool)]
static extern bool IsWindowVisible(IntPtr hWnd);

// FFISlice -> ReadOnlySpan<byte> : zero copie.
// (Demo en memoire : on cree un faux slice qui pointe vers un tableau manage.)
byte[] data = { 1, 2, 3, 4, 5, 6, 7, 8 };
FFISlice slice;
unsafe
{
    fixed (byte* p = data)
    {
        slice = new FFISlice { Ptr = (IntPtr)p, Len = (UIntPtr)data.Length };
    }
}

// Reconstruction managee zero-copie via unsafe + Span.
// (ReadOnlySpan<T> etant un ref struct, on l'utilise localement avec unsafe.)
unsafe
{
    ReadOnlySpan<byte> span = new(slice.Ptr.ToPointer(), (int)slice.Len);
    Console.WriteLine($"FFISlice: Ptr=0x{slice.Ptr.ToInt64():X}, Len={slice.Len}");
    Console.WriteLine($"ReadOnlySpan<byte>: Length={span.Length}, first byte={span[0]}");
}
Console.WriteLine("Marshaling blittable confirme : zero copie entre natif et manage.");
Console.WriteLine("Piege du bool : [MarshalAs(UnmanagedType.Bool)] = 4 octets cote Win32.");

FFISlice: Ptr=0x27826325680, Len=8


ReadOnlySpan<byte>: Length=8, first byte=1


Marshaling blittable confirme : zero copie entre natif et manage.


Piege du bool : [MarshalAs(UnmanagedType.Bool)] = 4 octets cote Win32.


## Exercices (stubs C.1 - a completer par l'etudiant)

Les trois exercices ci-dessous sont des **stubs** : ils s'executent sans erreur (conformement a la regle C.1 - *pas d'erreur volontaire*), mais le travail de l'etudiant est de remplacer le `TODO` par la logique demandee. Chaque enonce est calibre sur un des analyseurs Roslyn d'`AgentGuard`.

### Exercice 1 - Le piege du `bool` dans un struct partage

L'etudiant recoit un struct qui corrompt silencieusement les donnees natives. Le symptome : une methode native renvoie `true` mais la valeur vue cote C# est `false`. **Cause** : marshaling du `bool` par defaut (1 octet) au lieu de `UnmanagedType.Bool` (4 octets). L'exercice : corriger l'annotation et verifier que le resultat est desormais correct.

In [7]:
// Stub Exercice 1 - l'etudiant doit corriger l'annotation du champ Visible.
// Indice : ajouter [MarshalAs(UnmanagedType.Bool)] sur le champ pour matcher la
// convention Win32 (4 octets) au lieu du defaut C# (1 octet).
struct Point
{
    public int X;
    public bool Visible; // <- TODO : annoter pour matcher le natif (4 octets Win32)
    public int Y;
}

var p = new Point { X = 10, Visible = true, Y = 20 };
Console.WriteLine($"Point: X={p.X}, Visible={p.Visible}, Y={p.Y}");
Console.WriteLine("Exercice a completer : corriger l'annotation du bool.");

Point: X=10, Visible=True, Y=20


Exercice a completer : corriger l'annotation du bool.


### Exercice 2 - Reproduire la famine

La cellule §4 expose **deux leviers** distincts, et ils ne produisent pas la même mesure. L'étudiant en retire un, relance la cellule, et **mesure** : les deux résultats qualitatifs sont donnés ci-dessous, la quantification (médiane et dispersion sur les 5 runs, comme §4) est le travail demandé.

- **Retirer `TaskContinuationOptions.ExecuteSynchronously`** : la continuation n'est plus autorisée à s'exécuter en ligne, elle part sur le pool managé, et le temps de blocage du thread natif **disparaît des deux configurations** — la famine devient impossible. Mais ce n'est pas le levier de l'article ScyllaDB : on a supprimé l'exécution en ligne, pas la protection.
- **Retirer `TaskCreationOptions.RunContinuationsAsynchronously`** : c'est la protection qui tombe. La continuation s'exécute en ligne sur le thread natif, et le blocage **revient** — les deux lignes du tableau de §4 repassent du côté de la famine.

**Question de synthèse** : pourquoi retirer `ExecuteSynchronously` n'est-il pas une protection suffisante en production, alors que c'est lui qui rend la famine possible ? Qui contrôle les continuations attachées à la tâche, quand plusieurs bibliothèques la partagent ?

**Indice de manipulation** : la cellule §4 prend **un** levier en paramètre (`runAsync` : présence ou non de `RunContinuationsAsynchronously`) ; l'autre (`ExecuteSynchronously`) est fixé dans l'appel `ContinueWith` — le retirer demande d'éditer la cellule avant de la relancer.

In [8]:
// Stub Exercice 2 - l'etudiant doit rejouer la mesure de §4 avec un autre levier.
// Indice : seul `runAsync` est un parametre de §4 ; `ExecuteSynchronously` est
// fixe dans l'appel `ContinueWith` - editer la cellule de §4 avant de relancer.
// Rejouer avec
// (a) `TaskContinuationOptions.ExecuteSynchronously` retire, puis
// (b) `TaskCreationOptions.RunContinuationsAsynchronously` retire,
// et commenter pour chaque cas ce qui change dans le temps de blocage mesure
// du thread natif (mediane + min/max sur 5 runs, comme §4).
Console.WriteLine("Exercice 2 : rejouer §4 en changeant un levier, et commenter la mesure.");

Exercice 2 : rejouer §4 en changeant un levier, et commenter la mesure.


### Exercice 3 - Plomber une annulation a travers la frontiere

Le TCB de §3 n'a que deux canaux (`success`, `fail`). L'exercice : etendre la struct en un canal d'annulation explicite C# -> natif, sur le modele `CancellationTokenPropagation`. La signature devient : `(success, fail, cancel)` ou `cancel` est un pointeur de fonction appele par le code manage quand le `CancellationToken` est fired.

In [9]:
// Stub Exercice 3 - etendre AsyncCallbackHandle avec un canal cancel.
// Indice : ajouter un champ IntPtr CancelCallback, et la plomberie cote C#
// qui appelle ce callback quand CancellationToken est fired.
[StructLayout(LayoutKind.Sequential)]
struct AsyncCallbackHandleEx
{
    public IntPtr SuccessCallback;
    public IntPtr FailCallback;
    // TODO : ajouter le pointeur cancel et la plomberie CancellationToken
    public IntPtr UserData;
}

Console.WriteLine("Exercice 3 : etendre la struct avec le canal cancel et la plomberie.");

Exercice 3 : etendre la struct avec le canal cancel et la plomberie.


## Conclusion - la frontiere comme lieu d'enseignement

Ce que cet article ScyllaDB rend visible, c'est que les garde-fous appris dans `06-Aspire-GardeFous-Roslyn` (analyseurs Roslyn qui detectent les blessures classiques du code d'agent) **redeviennent pertinents a la frontiere native** :

- `.Result` qui deadlock en manage -> callback de completion qui famine en FFI. **Meme physique**, meme fix : `await` cote manage, `RunContinuationsAsynchronously` cote FFI.
- `AsyncVoid` qui crash le process manage -> exception non marshallee qui corrompt l'etat natif. **Meme physique**, meme fix : un canal d'erreur explicite.
- `CancellationTokenPropagation` perdu -> operation qui ne s'annule pas et bloque un thread natif pour toujours. **Meme physique**, meme fix : plomberie explicite.

La frontiere n'est pas un endroit ou les bonnes pratiques s'arretent. C'est un endroit ou **elles deviennent critiques**, parce que le ramasse-miettes et le runtime ne peuvent plus vous rattraper.

## References

- ScyllaDB (2026-08-31). *Self-Baked Async FFI Framework for Rust <-> C# Interop*. https://www.scylladb.com/2026/08/31/async-ffi-framework-for-rust-c-interop/
- Notebook ancre : `MyIA.AI.Notebooks/GenAI/Integrations-DotNet/Aspire/06-Aspire-GardeFous-Roslyn.ipynb` (analyseurs `AgentGuard.Analyzers`).
- Docs .NET : `UnmanagedCallersOnlyAttribute`, `delegate* unmanaged`, `GCHandle`, `SafeHandle`, `TaskCompletionSource.RunContinuationsAsynchronously`.
**Suite** : [09 — le harness Copilot SDK →](09-Aspire-Harness-CopilotSdk.ipynb) — le harness agent lui-même, exercé pour de vrai : auth réelle, catalogue de modèles, tours de conversation, cycle de vie `SessionEvent` observable.